In [1]:
import pandas as pd
import nltk
import re
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

english_stopwords = set(stopwords.words('english'))

In [2]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     /Users/upanshuparekh/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/upanshuparekh/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/upanshuparekh/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/upanshuparekh/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /Users/upanshuparekh/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagg

True

# Data Preprocessing -- Reddit
In this part (2), I will be using [NLTK](https://www.nltk.org)'s data preprocessing functions to clean up the data produced by the `data_collection.py` script. This data must then be somehow plugged into sentiment analysis libraries to get some score.

Here's what I'll do. For each post:
- Get a sentiment score for the post title + self text, call it $s_0$
- Get a sentiment score for each top comment, call them $s_1, s_2, ..., s_{10}$
- Get a weighted aggregate sentiment score for the post. Weighing a score $s_i$ less as $i$ increases. I will just begin with the simple function: $w(s_i) = 1/i$ to multiply to a score to weigh it.

For each month's of post sentiment scores, I will take the median score as the representation for the entire month, as this statistic is more resistant to outliers.

I have to do it like this because sentiment analysis really starts to break when the text gets too long, either with NLTK's VADER, or with FinBERT. You will see this in my previous commits if you want to look.

So all the data preprocessing will preserve the fields necessary to conduct this analysis.

In [2]:
df = pd.concat([
    pd.read_csv('reddit-cryptocurrency-data.csv'),
    pd.read_csv('reddit-wallstreetbets-data.csv'),
    pd.read_csv('reddit-finance-data.csv'),
    pd.read_csv('reddit-investing-data.csv'),
], ignore_index=True)

In [3]:
df.columns

Index(['subreddit', 'month', 'post_id', 'post_title', 'post_selftext', 'tc0',
       'tc1', 'tc2', 'tc3', 'tc4', 'tc5', 'tc6', 'tc7', 'tc8', 'tc9'],
      dtype='object')

In [4]:
df.sample(5)

,subreddit,month,post_id,post_title,post_selftext,tc0,tc1,tc2,tc3,tc4,tc5,tc6,tc7,tc8,tc9
4594,investing,Oct,1gcqz72,It will probably not come as an enormous surpr...,&gt;We study a sample of 513 reports and find ...,Sell side will use the quick method that gives...,It generates brokerage commissions yes.,"Even if they use a DCF, a DCF is only as good ...","...Yes, but... does it work?",A valuation metric on any company requires car...,interesting sell side analysts are not compen...,That's why you would use an inverse DCF model....,You do understand that creating a decent DCF m...,You can build an inverse DCF model to determin...,[removed]
4110,investing,Jun,1df21ja,If Musk gets his 56b then what?,Prefer not to speculate on how much Musk has g...,It is supposed to “encourage” his attention to...,The promise of the package is what was to enco...,As the judge in the case he lost pointed out: ...,Nothing happens.,I'm not sure what Musk now offered to the Tesl...,Most of teslas value is tied to hype around El...,56 billion flows out of tesla and into elon's ...,The only thing relevant here is the cult of pe...,"&gt;Personally, as a shareholder I think it's ...",I don't even think it's that now. All the inve...
4644,investing,Nov,1gz9hf4,Have you ever been so afraid of a market drop ...,"For the first time, I’ve been able to save a s...",Oh my. Analysis paralysis. You’re hurting your...,I am in a similar situation with similar numbe...,“Risk” is completely defined by an investor’s ...,How many times can one contradict themself in ...,"People said Apple, the S&amp;P 500, etc. were ...",Read up on [the world's worst market timer](ht...,I think the more worrying thing is if you inve...,If your horizon is long it simply doesn’t matt...,Obviously actually look at the about. Continue...,Yes. I saw covid coming and pulled out of the...
2123,wallstreetbets,Oct,1g0z458,Insurance companies in FL right now,[deleted],They be fine most people don’t have flood insu...,https://preview.redd.it/5wq48z39f1ud1.jpeg?wid...,"Flood insurance is separate, expensive, and ha...",“Oh you have standard flood insurance? Unfortu...,I do residential insurance work. Specifically ...,"Oh, I'm sorry, the hurricane flooding insuranc...",It wasn't a flood. The ocean leaked on my house.,"I mean, if you don't read the legally binding ...",Mother nature blowing up the Ponzi scheme,"Nah, this is about to be the reinsurers that a..."
246,cryptocurrency,Mar,1bhsisr,Realised today that I don't like where Ethereu...,Background : I only hold BTC and ETH (70%/30...,Someone will make a killer app or they won't. ...,Loopring Wallet has pretty much done this.,then they seriously need to work on their bran...,"The ethereum devs, like the bitcoin devs, real...",You are absolutely right. Crypto is a bit up i...,LRC is exactly that. Correct.,L2 are a way to scale Ethereum. There is no m...,remember when you had to download the entire b...,I don’t think you understand. BTC is a store...,Here’s a news flash. Crypto in general is not ...


In [5]:
# Combine `post_title`, `post_selftext` into a column 'headline'
cols_to_combine = ['post_title', 'post_selftext']
df['headline'] = df[cols_to_combine].fillna('').agg(' '.join, axis=1)
df = df.drop(columns=cols_to_combine)

In [6]:
df.sample(5)

,subreddit,month,post_id,tc0,tc1,tc2,tc3,tc4,tc5,tc6,tc7,tc8,tc9,headline
3062,finance,Jul,1dtzz68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,US FDIC issues consent order to Thread Bank FD...
1128,cryptocurrency,Dec,1hfbq9a,"Big deal, I can turn that into 0 in no time.","tldr; A Solana trader lost $74,000 in three mi...",&gt;A cryptocurrency trader has lost nearly $7...,So dumb greedy trader flipped a coin and chose...,A crypto legend spotted!!!. You are doing wel...,A common mistake.,"Buy high sell low, my goated strategy.","Another whale saw his buy, dumped in to it. Sa...","Degenerate move, nicely done",We are so back,Solana trader turns $112k into $38k in three m...
1344,wallstreetbets,Feb,1ataqju,Cash out some winnings and invest in a higher ...,860k left in SPY would be more than 1.2M today...,"Bitch you had $860,000 to invest, disingenuous...",Lol at having to explain print screen to someo...,Lol. The fidelity app doesn't allow me to see ...,"“Print screen” is a great feature, crop it, th...",Step one. Be rich,"Exactly, you could have left it in any ETF and...",money can't buy brains. not saying OP is dumb.,LMAO right. Dude gotta be trolling.,860k &gt;&gt; 500k &gt;&gt; 1.2mil It's been a...
3445,finance,Nov,1gqo72e,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Agicap Secures €45 Million in Series C Funding...
910,cryptocurrency,Oct,1fvyalo,What I love about the internet and human natur...,But his statement is still right. Bananas are ...,Typical post by rizzobitcoin. Annoying useless...,Indeed. ![gif](giphy|9wLKh6ms5t9qE),Well we cant put Bitcoin in our butthole so th...,Spot on. Mark has certainly changed his stance...,You could get more than 6x in other assets and...,No But it did now,Duh he’s already rich. Poors can’t imagine why...,Too many stupid comments,Mark Cuban saying he'd buy bananas over Bitcoi...


## Text Cleaning Function
I did research the tradeoff between stemming vs. lemmatizing, and in general I got that:
- Stemming = rules-based, heuristic algorithmic removal of common word endings
    - faster for larger datasets, loses accuracy and context, can produced nonexistent words
- Lemmatizing = more accurate, more computationally expensive with Part-of-Speech Tagging required

But I reason that I'm not training an ML model where accuracy is mission critical, so simply
stemming should suffice.

In [7]:
def clean_text_stemmer(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra whitespace, and using
    NLTK's tokenization, stopword removal, stemming.
    """
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove special characters and digits, keep important punctuation
    text = re.sub(r'[^A-Za-z\s.,!?]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip().lower()

    words = word_tokenize(text)
    # Remove stopwords
    words = [word for word in words if word not in english_stopwords]
    # Stemming
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    text = ' '.join(words)
    return text + '.'

In [13]:
example_text = df['headline'].sample(1).values[0]
example_text

'Parents scammed out of 30k in crypto As the title says my parents got scammed out of 30k in crypto but we’re not sure how exactly the guy got access to their wallets.   My dad downloaded the app Zoho on his phone but his Coinbase wallet was on his computer and somehow the guy gained access to all of his wallets on his computer without ever downloading zoho onto his actual computer. How is this possible? My dad never gave them his usernames or passwords or seeds or anything and somehow the guy got into their account. Has anybody had experience with this?'

In [14]:
example_text_stemmed = clean_text_stemmer(example_text)
example_text_stemmed

'parent scam k crypto titl say parent got scam k crypto sure exactli guy got access wallet . dad download app zoho phone coinbas wallet comput somehow guy gain access wallet comput without ever download zoho onto actual comput . possibl ? dad never gave usernam password seed anyth somehow guy got account . anybodi experi ?.'

Okay, maybe lemmatizing is the better strategy, there are just too many nonsense words here that can throw off the sentiment analyzer.

In [15]:
def clean_text_lemmatizer(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra whitespace, and using
    NLTK's tokenization, stopword removal, lemmatizer.
    """
    # POS tagging for lemmatization
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    pos_tags = pos_tag(word_tokenize(text))
    lemmatizer = WordNetLemmatizer()
    # Map POS tags to WordNet format
    tag_dict = {
        "J": wordnet.ADJ,  # Adjective
        "N": wordnet.NOUN, # Noun
        "V": wordnet.VERB, # Verb
        "R": wordnet.ADV   # Adverb
    }
    pos_tags = [(word, tag_dict.get(tag[0], 'n')) for word, tag in pos_tags]

    # Lemmatization
    words = [lemmatizer.lemmatize(word, pos).lower() for word, pos in pos_tags]
    # Remove stopwords
    words = [word for word in words if word not in english_stopwords]
    text = ' '.join(words)

    # Remove special characters and digits, keep important punctuation
    text = re.sub(r'[^A-Za-z\s.,!?]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip().lower()

    return text + '.'  # For FinBERT

In [16]:
example_text

'Parents scammed out of 30k in crypto As the title says my parents got scammed out of 30k in crypto but we’re not sure how exactly the guy got access to their wallets.   My dad downloaded the app Zoho on his phone but his Coinbase wallet was on his computer and somehow the guy gained access to all of his wallets on his computer without ever downloading zoho onto his actual computer. How is this possible? My dad never gave them his usernames or passwords or seeds or anything and somehow the guy got into their account. Has anybody had experience with this?'

In [17]:
example_text_lemmatized = clean_text_lemmatizer(example_text)
print('LEMMATIZED: ' + example_text_lemmatized)
print('STEMMED: ' + example_text_stemmed)

LEMMATIZED: parents scammed k crypto title say parent get scammed k crypto sure exactly guy get access wallet . dad download app zoho phone coinbase wallet computer somehow guy gain access wallet computer without ever download zoho onto actual computer . possible ? dad never give usernames password seed anything somehow guy get account . anybody experience ?.
STEMMED: parent scam k crypto titl say parent got scam k crypto sure exactli guy got access wallet . dad download app zoho phone coinbas wallet comput somehow guy gain access wallet comput without ever download zoho onto actual comput . possibl ? dad never gave usernam password seed anyth somehow guy got account . anybodi experi ?.


## Applying Lemmatizer to Whole Text Column
Okay the lemmatizer **definitely works a lot better**, it's a whole lot more accurate in its processing. Gonna stick with that! Now to apply it to the whole text column.

In [7]:
# For some reason there are floats in the text data, have to replace those with empty string
cols = ['headline'] + [f'tc{i}' for i in range(10)]
df[cols] = df[cols].astype(str)

In [53]:
for col in cols:
    df[col] = df[col].apply(clean_text_lemmatizer)

In [54]:
df.sample(4)

,subreddit,month,post_id,tc0,tc1,tc2,tc3,tc4,tc5,tc6,tc7,tc8,tc9,headline
4516,investing,Oct,1fu1qjr,low go ? hysa practically zero decade . statis...,. keep enough cash cash hysa emergency month g...,japan negative nominal interest rate decade bo...,"invest general recommendation , lose . , world...",view hysa investment tool . . place safely sto...,good lesson recency bias ..,"get boat op , invest instead put away account ...","sofi still . , think spaxx ..",case nt consider . inflation around right want...,react . hysa rate fall vacuum . low risk optio...,"react fall hysa interest rate ? week get pay ,..."
234,cryptocurrency,Mar,1bc9lsq,hope theoretical dip happen nontheoretical mon...,m curious see etfs cushion drop somewhat peopl...,"hate say time s different s lie , love say , m...",cash ready . working hour take overtime much p...,according every crypto influencer yt s hour le...,"er , re call dip , re call buying opportunitie...","time definitely feels different , least regard...","s theory bitcoin bull cycle cause halvings , r...","yea , price go ath month halving crazy . alone...",lol.yes samesie ..,ready big dip even bull market . s unusual see...
2998,finance,Jun,1d9pvxw,nan.,nan.,nan.,nan.,nan.,nan.,nan.,nan.,nan.,nan.,afford ? remove.
804,cryptocurrency,Sep,1fi1wne,show btc.,one go check headline claim ... year ago today...,remember bullish it immediately think send eth...,benjamin cowen enter chat .....,"alt like , since last bull run.",one ? nt bullrun since.,bitcoin go k dec k march . call bull run ?.,moment talk bitcoin dominance ?.,"thanks , make sense ..",make sense make clear ..,"year ago today , eth officially complete longa..."


In [55]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('reddit-cleaned.csv', index=False)